# CLM-0.3c — Counterfactual Mitosis

Formal Kaggle runner for the one-shot counterfactual WHEN/WHERE experiment. Enable two T4 GPUs before setting `RUN_FORMAL = True`.

In [ ]:
import subprocess, sys
from pathlib import Path

ROOT = Path('/kaggle/working/mini-cells')
BRANCH = 'codex/clm-0.3c-counterfactual-mitosis'
REPO = 'https://github.com/ArcheLabs/mini-cells.git'

if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO, str(ROOT)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=ROOT, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=ROOT, check=True)

HEAD = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=ROOT, text=True).strip()
print('HEAD:', HEAD)


In [ ]:
import torch
print({
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'gpu_count': torch.cuda.device_count(),
    'gpus': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],
})
assert torch.cuda.device_count() >= 1, 'CUDA is required'


In [ ]:
# Mandatory CPU/static preflight before spending GPU time.
subprocess.run([
    sys.executable, '-m', 'pytest',
    'tests/test_clm_counterfactual_mitosis.py',
    'tests/test_growth_router.py',
    'tests/test_growth_checkpoint.py',
    '-q',
], cwd=ROOT, check=True)


In [ ]:
RESULTS = ROOT / 'results/clm-0.3c-counterfactual-mitosis'
RUN_FORMAL = False
RESTART_EXISTING = False  # only set True after a code-commit change; it deletes old 0.3c evidence

cmd = [
    sys.executable,
    'scripts/run_clm_counterfactual_mitosis_003.py',
    '--output-root', str(RESULTS),
]
if RESTART_EXISTING:
    cmd.append('--restart-existing')
if RUN_FORMAL:
    cmd.append('--execute')
subprocess.run(cmd, cwd=ROOT, check=True)


In [ ]:
# After the formal run succeeds, inspect the machine-readable decision.
import json
decision_path = RESULTS / 'decision.json'
if decision_path.exists():
    print(json.dumps(json.loads(decision_path.read_text()), indent=2, sort_keys=True))
else:
    print('decision.json not produced yet')


In [ ]:
PUBLISH = False
if PUBLISH:
    subprocess.run([
        sys.executable,
        'scripts/publish_clm_0_3c_results.py',
        '--push',
    ], cwd=ROOT, check=True)
